# Partial cross entropy on SpaceNet buildings

Runtime > Change runtime type > T4 GPU

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available())

In [ ]:
!git clone -b claude/human-code-style-tewbfy https://github.com/iMohamedMamdouh/Task1.git
%cd Task1
!pip install -q rasterio

## Data

In [ ]:
!python -m scripts.download_spacenet --num-tiles 900 --workers 32

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

from pointseg.data import sample_points

tiles = Path('data/rio/tiles.txt').read_text().split()
mask = (np.array(Image.open(f'data/rio/masks/{tiles[0]}')) > 127).astype('uint8')
points = sample_points(mask, 5, 'balanced', np.random.default_rng(0))

print(len(tiles), 'tiles')
print(int((points != 255).sum()), 'labelled pixels of', points.size)

## Loss

In [ ]:
!python -m pytest tests -q

## Training

In [ ]:
!python -m pointseg.train --points 5 --epochs 40 --batch-size 16 --device cuda --out runs/points5_ce.json

## Experiment grid

Point density 1 / 5 / 20 / 100, full masks, focal gamma 0 and 2, two repeated seeds.
Finished runs are skipped when the cell is run again.

In [ ]:
!python -m scripts.run_experiments --epochs 40 --batch-size 16 --device cuda --workers 1 --save-checkpoints

## Results

In [ ]:
!python -m scripts.make_figures --checkpoint-dir runs

In [ ]:
from IPython.display import Image as Show, display

for name in ['annotation_example', 'label_efficiency', 'training_curves', 'qualitative']:
    path = Path('report/figures') / f'{name}.png'
    if path.exists():
        display(Show(str(path)))

In [ ]:
import pandas as pd

pd.read_csv('report/figures/results.csv')

## Save to Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
backup = Path('/content/drive/MyDrive/pointseg')
backup.mkdir(parents=True, exist_ok=True)

!cp -r runs report/figures {backup}/